# Inference Configuration
Specify
- path

In [10]:
path = "../../output/protenn2/v5"


In [11]:
import json
import os.path
import pickle

from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from src.protenn2.utils import get_train_val_test_paths, get_device

# get file paths

log_path = os.path.join(path, "log")

label_encoder_path = os.path.join(path, "label_encoder.pkl")
model_path = os.path.join(path, "best_model.pt")
with open(os.path.join(path, "params.json"), "r") as f:
    params = json.load(f)
if "input_folder" not in params:
    raise ValueError("input_folder must be specified")
dataset_path = os.path.join("../../", params["input_folder"])
train_path, val_path, test_path = get_train_val_test_paths(dataset_path)


In [12]:
from src.protenn2.utils import calculate_max_protein_length
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

# Initialize objects

device = get_device()
with open(label_encoder_path, "rb") as f:
    label_encoder: LabelEncoder = pickle.load(f)
num_classes = len(label_encoder.classes_)
max_protein_length = calculate_max_protein_length(dataset_path)

test_dataset = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                         embedding_dir="../../data/embeddings/protein_embeddings_new")

collate_fn = create_protein_collate_fn(max_protein_length, test_dataset.padding_encoded_id)

test_dataloader = DataLoader(test_dataset, collate_fn=collate_fn)

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

Using MPS (Apple Silicon GPU).
Max protein length: 599
Dataset initialized with 1317 unique proteins.


In [13]:
import numpy as np

test_dataset2 = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                          embedding_dir="../../data/embeddings/protein_embeddings_new")
y_true_labels_list = []
for _, y_true_labels, _ in test_dataset2:
    y_true_labels_list.append(y_true_labels)


y_true_flat = np.concatenate(y_true_labels_list)
classes, counts = np.unique(y_true_flat, return_counts=True)
global_distribution = np.zeros(test_dataset.num_classes)
global_distribution[classes] = counts / counts.sum()

Dataset initialized with 1317 unique proteins.


In [14]:
# y_true_labels_list_c = [mapper.map_labels(labels, "C") for labels in y_true_labels_list]
# y_true_flat = np.concatenate(y_true_labels_list_c)
# classes, counts = np.unique(y_true_flat, return_counts=True)
# global_distribution = np.zeros(mapper.get_class_count("C"))
# global_distribution[classes] = counts
# global_distribution

In [15]:
dummy_stratified_classifier_per_protein_kwargs = {"distribution": global_distribution}
dummy_majority_classifier_per_protein_kwargs = {"majority_label": 1}

In [16]:
from src.protenn2.analysis.inference import run_inference_dummy, dummy_stratified_classifier_per_protein, \
    dummy_majority_classifier_per_protein

y_true_labels_list, y_pred_confidences_list, protein_chain_id_list = run_inference_dummy(
    dummy_classifier=dummy_majority_classifier_per_protein,
    dataloader=test_dataloader,
    padding_encoded_id=test_dataset.padding_encoded_id,
    num_classes=mapper.get_class_count(target_hierarchy="H"),
    dummy_classifier_kwargs=dummy_majority_classifier_per_protein_kwargs)

Running dummy inference on 1317 proteins...


Dummy Inference Progress: 100%|██████████| 1317/1317 [00:01<00:00, 1215.50it/s]

Dummy inference complete. Processed 1317 proteins.


# Analysis Configuration

In [17]:


bootstrap_samples = 1000
post_process_kwargs = None
post_process_func = None
metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
                      "segment_overlap_score")
# metrics_to_compute = ("f1_score", "recall_score", "precision_score")

In [21]:
y_pred_confidences_list

[array([[0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]]),
 array([[0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]]),
 array([[0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]]),
 array([[0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]]),
 arr

In [18]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=None,
                                                post_process_kwargs=None,
                                                metrics_to_compute=metrics_to_compute,
                                                levels=("D"),
                                                name="baseline_majority")

---- Computing Metrics for hierarchy: D
('accuracy', 'f1_score', 'jaccard_score', 'recall_score', 'precision_score', 'segment_overlap_score')
----- Accuracy Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:01<00:00, 687.46it/s]


{'mean': np.float64(0.6681649253574012), 'ci_lower': np.float64(0.6497547633868724), 'ci_upper': np.float64(0.6879753601928632), 'alpha': 0.05}
----- F1 Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:07<00:00, 138.69it/s]


{'mean': np.float64(0.8010358358334024), 'ci_lower': np.float64(0.7876986037599985), 'ci_upper': np.float64(0.8151485814390026), 'alpha': 0.05}
----- Jaccard Score Metrics -----


...

{'mean': np.float64(1.0), 'ci_lower': np.float64(1.0), 'ci_upper': np.float64(1.0), 'alpha': 0.05}
----- Precision Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:06<00:00, 142.94it/s]


{'mean': np.float64(0.6681649253574012), 'ci_lower': np.float64(0.6497547633868724), 'ci_upper': np.float64(0.6879753601928632), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:51<00:00, 19.48it/s]

{'mean': np.float64(0.9011912320375564), 'ci_lower': np.float64(0.8919651222493007), 'ci_upper': np.float64(0.9112033899339275), 'alpha': 0.05}


# All results

In [19]:
all_results

{'baseline_majority_D': {'accuracy': {'mean': np.float64(0.6681649253574012),
   'ci_lower': np.float64(0.6497547633868724),
   'ci_upper': np.float64(0.6879753601928632),
   'alpha': 0.05},
  'f1_score': {'mean': np.float64(0.8010358358334024),
   'ci_lower': np.float64(0.7876986037599985),
   'ci_upper': np.float64(0.8151485814390026),
   'alpha': 0.05},
  'jaccard_score': {'mean': np.float64(0.6681649253574012),
   'ci_lower': np.float64(0.6497547633868724),
   'ci_upper': np.float64(0.6879753601928632),
   'alpha': 0.05},
  'recall_score': {'mean': np.float64(1.0),
   'ci_lower': np.float64(1.0),
   'ci_upper': np.float64(1.0),
   'alpha': 0.05},
  'precision_score': {'mean': np.float64(0.6681649253574012),
   'ci_lower': np.float64(0.6497547633868724),
   'ci_upper': np.float64(0.6879753601928632),
   'alpha': 0.05},
  'segment_overlap_score': {'mean': np.float64(0.9011912320375564),
   'ci_lower': np.float64(0.8919651222493007),
   'ci_upper': np.float64(0.9112033899339275),
   '

In [20]:
with open(os.path.join(path, "baseline_test_metrics_majority.json"), "w") as f:
    json.dump(all_results, f)